[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws25/blob/main/expectation/markov_models_melodic_expectation.ipynb)

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

HOME_DIR = "."

if IN_COLAB:
    !pip install partitura
    !pip install python-hiddenmarkov
    
    !git clone --recurse-submodules https://github.com/MusicalInformatics/miws25
    import sys
    sys.path.insert(0, "/content/miws25/expectation/")
    HOME_DIR = "/content/miws25/expectation/"

# Melodic Expectation with Markov Models

In this notebook we will look at Markov Chains for modeling musical expectation.

## Quick Recap of Probability

* Probability: measure of the **likelihood** of an event

* $0\leq p(x) \leq 1$, 
    * $p(x) = 0$ indicates that the event is very unlikely to occur 
    * $p(x) = 1$ indicates that the event will most likely occur
    
* Random variables: $X$ can take values $x$

    
## Quick Recap: Graph Theory

* Graphs: A tuple $\mathcal{G}=(\mathbf{X}, \mathbf{E})$ consisting in a set of nodes $\mathbf{X} = \{X_1,\dots,X_N\}$ and a set of edges $\mathbf{E}$ connecting the nodes.

* Edges can be *directed* $(X_i \rightarrow X_j)$ or *undirected* $(X_i - X_j)$. If the nodes of a graph are directed, we call it a *directed Graph*, otherwise is an *undirected graph*
* Parent and Children nodes: In directed graphs, if $(X_i \rightarrow X_j)\in \mathbf{E}$ then $X_i$ is a parent of $X_j$ and $X_j$ is a child of $X_i$. 
    * $\mathbf{Pa}(X_i)$ is the set of parents of $X_i$
    * $\mathbf{Ch}(X_i)$ is the set of children of $X_i$
    
Let's consider this graph

<div>
<img src="img/graph_example.png" width="250"/>
</div>

* the parents of $X_3$ are $\mathbf{Pa}(X_3) = \{X_1, X_2\}$.
* Which are the parents of $X_4$?
* Which are the parents of $X_1$?


## Quick Recap: Probabilistic Graphical Models

* Probabilistic graphical models (PGMs) provide a way to visualize the structure of a probabilisitc model

* Easy and elegant way to represent conditional independence properties

* **Bayesian Networks**: Nodes in a graph represent *random variable* and edges specify conditional independence properties:

$$p(X_1, \dots, X_N) = \prod_{i=1}^{N} p(X_i \mid \mathbf{Pa}(X_i))$$


For the example above

$$p(X_1, X_2, X_3, X_4, X_5, X_6) = p(X_1) p(X_2) p(X_3\mid X_1, X_2) p(X_4 \mid X_3) p(X_5 \mid X_3) p(X_6 \mid X_3)$$

## Markov Models

### Independent Observations
* The simplest way of modeling a sequence of observations is to treat them as independent

<div>
<img src="img/independent_markov.png" width="250"/>
</div>

$$p(\mathbf{x}_1\dots, \mathbf{x}_N) = \prod_{n=1}^{N} p(\mathbf{x}_i)$$

but this is a poor assumption for inherently sequential data (like music!)

### First order Markov Chains

The conditional distribution of each variable is independent of all previous observations except for the *most recent*

<div>
<img src="img/first_order_markov.png" width="250"/>
</div>

$$p(\mathbf{x}_1, \dots, \mathbf{x}_N) = p(\mathbf{x}_1)\prod_{n=2}^{N}p(\mathbf{x}_n \mid \mathbf{x}_{n-1})$$



## Information theoretic measures of musical expectation

* Information Content: Unexpectedness of a musical event

$$\text{IC}(\mathbf{x}_n\mid \mathbf{x}_{n-1}, \dots) = -\log_2 p(\mathbf{x}_n \mid \mathbf{x}_{n-1}, \dots )$$

* Entropy: How uncertain is a musical event

$$\text{H}(\mathbf{x}_{n-1:1})= \sum_{i \in \mathbf{A}} p(\mathbf{x}_i \mid \mathbf{x}_{n-1}, \dots) \text{IC}(\mathbf{x}_i\mid \mathbf{x}_{n-1}, \dots)$$

where $\mathbf{A}$ is the set of possible states that $\mathbf{x}$ can take.



In [ ]:
import os

# Uncomment this line if the kernel keeps crashing
# See https://stackoverflow.com/a/53014308
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

import numpy as np
import partitura as pt
import matplotlib.pyplot as plt

%config InlineBackend.figure_format ='retina'

In [ ]:
from utils import load_data

# To filter out short melodies by the minimum number of notes that a sequence should have

min_seq_len = 10
sequences = load_data(min_seq_len)

## Tasks 1: Data loading & preparation
1. Check out the content of the variable "sequences", if unclear have a look at the loading function.

2. Which musical texture do these sequences exhibit? (https://en.wikipedia.org/wiki/Texture_(music))

3. Write a function to derive sequences of pitches from this data.

4. Write a function to derive sequences of durations from this data. Modify this to compute inter onset intervals (IOIs; the time between two consecutive onsets). Can you encode rests as well by comparing duration with IOI? 

In [ ]:
# Decide which features to use!
feature_names = "pitch"
features = []
for seq in sequences:

    if feature_names == "pitch":
        # pitch
        features.append(seq["pitch"])
    elif feature_names == "pitch_class":
        # pitch class
        features.append(np.mod(seq["pitch"], 12))
    elif feature_names == "pitch_duration":
        # pitch and duration
        features.append(np.column_stack([seq["pitch"], seq["duration_sec"]]))
    elif feature_names == "pitch_ioi":
        # TODO: Implement computing IOIs!
        pass

In [ ]:
from typing import List
from sklearn.preprocessing import LabelEncoder
from itertools import product
from utils import get_indices_cartesian_product, find_nearest, QUANTIZED_DURATIONS


class Encoder(object):
    """
    A hand-crafted encoder
    """

    def __init__(self, feature_names):
        self.feature_names = feature_names
        self.classes_ = None
        self.dim_encoders: List[LabelEncoder] = []

    def fit(self, y: np.ndarray) -> None:

        if self.feature_names in ("pitch", "pitch_class"):
            pitch_dim_enc = LabelEncoder()

            pitch_dim_enc.fit(y)

            self.dim_encoders = [pitch_dim_enc]

            self.classes_ = pitch_dim_enc.classes_

        else:
            # Assume first column is pitch and second is time related
            pitch_dim_enc = LabelEncoder()
            pitch_dim_enc.fit(y[:, 0])

            time_dim_enc = LabelEncoder()

            # MIDIs were generated at 100bpm
            quantized_time_in_beats = find_nearest(
                QUANTIZED_DURATIONS, np.round(y[:, 1] * 100 / 60, 3)
            )

            time_dim_enc.fit(quantized_time_in_beats)

            self.dim_encoders = [pitch_dim_enc, time_dim_enc]

            self.classes_ = np.array(
                list(product(pitch_dim_enc.classes_, time_dim_enc.classes_))
            )

    def transform(self, y: np.ndarray) -> np.ndarray:

        if self.feature_names in ("pitch", "pitch_class"):

            return self.dim_encoders[0].transform(y)

        else:

            pitch_encs = self.dim_encoders[0].transform(y[:, 0])
            time_encs = self.dim_encoders[1].transform(
                find_nearest(QUANTIZED_DURATIONS, np.round(y[:, 1] * 100 / 60, 3))
            )

            elem = np.column_stack([pitch_encs, time_encs])

            print(elem)

            return get_indices_cartesian_product(elem, self.classes_)

    def inverse_transform(self, y: np.ndarray) -> np.ndarray:

        if self.feature_names in ("pitch", "pitch_class"):

            return self.dim_encoders[0].inverse_transform(y)

        else:

            idxs = self.classes_[y.astype(int)]

            pitch_dim = self.dim_encoders[0].inverse_transform(idxs[:, 0])

            dur_quant = self.dim_encoders[1].inverse_transform(
                find_nearest(QUANTIZED_DURATIONS, np.round(idxs[:, 1] * 100 / 60, 3))
            )

            # Transform duration in seconds
            dur_dim = dur_quant * 60 / 100

            return np.column_stack([pitch_dim, dur_dim])

    def encode(self, y: np.ndarray) -> np.ndarray:

        return self.transform(y)

    def decode(self, y: np.ndarray) -> np.ndarray:

        decoded_labels = self.inverse_transform(y)

        if self.feature_names == "pitch":

            note_array = np.zeros(
                len(decoded_labels),
                dtype=[
                    ("pitch", int),
                    ("duration_sec", float),
                    ("onset_sec", float),
                    ("velocity", int),
                ],
            )

            note_array["pitch"] = decoded_labels

            note_array["onset_sec"] = np.arange(len(decoded_labels)) * 0.5

            note_array["duration_sec"] = 0.5

            note_array["velocity"] = 64

        elif self.feature_names == "pitch_class":
            note_array = np.zeros(
                len(decoded_labels),
                dtype=[
                    ("pitch", int),
                    ("duration_sec", float),
                    ("onset_sec", float),
                    ("velocity", int),
                ],
            )

            # All notes around Middle C
            note_array["pitch"] = decoded_labels + 60

            note_array["onset_sec"] = np.arange(len(decoded_labels)) * 0.5

            note_array["duration_sec"] = 0.5

            note_array["velocity"] = 64

        elif self.feature_names == "pitch_duration":
            pass

        elif self.feature_names == "pitch_ioi":
            pass

        return note_array

In [ ]:
QUANTIZED_DURATIONS[find_nearest(QUANTIZED_DURATIONS, 1)]

In [ ]:
enc = Encoder(feature_names)

if feature_names in ("pitch", "pitch_class"):
    # enc = LabelEncoder()
    enc.fit(np.hstack(features))
elif feature_names in ("pitch_duration"):
    # enc = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
    enc.fit(np.vstack(features))

encoded_sequences = [enc.transform(seq) for seq in features]

In [ ]:
enc.classes_

## Tasks 2: Data exploration:

1. Compute and draw a histogram of pitches. Modify this to show pitch classes!

2. Compute and draw a histogram of IOIs. The input MIDI files are deadpan, i.e. the IOIs in seconds correspond to the notated duration exactly. Look through the IOIs and make an educated guess for some smallest float time unit that could serve as integer smallest time division. Encode the IOIs as multiples of this smallest integer. Which multiples make musical sense?

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

alphabet_size = len(enc.classes_)
alphabet = enc.classes_
print(f"Number of sequences: {len(sequences)}")
print(f"Unique elements {alphabet_size}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    np.hstack(encoded_sequences),
    bins=alphabet_size,
    range=(0, alphabet_size),
    color="firebrick",
)
ax.set_xticks(np.arange(alphabet_size) + 0.5)
# ax.set_xticklabels(alphabet)
ax.set_ylabel("Count")
plt.show()

## Tasks 3: A Markov Chain

1. Choose a data type to model: pitch, pitch class, IOIs, or durations (including or without an encoding for rests). Concatenate all the sequences into one long data sequence.

2. You have now a sequence **X** of symbols from an alphabet **A** (set of possible symbols of your chosen data type):

$$ \mathbf{X} = \{\mathbf{x_0}, \dots, \mathbf{x_n} \mid \mathbf{x}_{i} \in  \mathbf{A} \forall i \in 0, \dots, n \}$$

Compute the empirical conditional probability of seeing any symbol after just having seen any other:

$$ p(\mathbf{x_i}\mid \mathbf{x_{i-1}}) $$

What is the dimensionality of this probability  given $\lvert A \rvert = d $?


In [ ]:
def maximum_likelihood_fom(sequences, alphabet_size):
    probs = np.zeros((alphabet_size, alphabet_size)) + 1e-6
    for seq in sequences:
        for p1, p2 in zip(seq[:-1], seq[1:]):
            probs[p1, p2] += 1
    probsum = np.sum(probs, axis=1)
    normalized_distribution = (probs.T / probsum).T
    return normalized_distribution


transition_probabilities = maximum_likelihood_fom(encoded_sequences, alphabet_size)
fig, ax = plt.subplots(figsize=(7, 7))
a = ax.matshow(
    transition_probabilities,
    aspect="equal",
    cmap="BuPu",
)
ax.set_xticks(range(alphabet_size))
# ax.set_xticklabels(enc.classes_)
ax.set_yticks(range(alphabet_size))
# ax.set_yticklabels(enc.classes_)

plt.colorbar(a)
plt.show()

In [ ]:
def probability_of_sequence(
    sequence,
    transition_probs=transition_probabilities,
    alphabet_size=alphabet_size,
):
    """
    p(x_1,...,x_n)
    """
    p_0 = 1 / alphabet_size
    transitions = np.prod(
        [transition_probs[i, j] for i, j in zip(sequence[1:], sequence[:-1])]
    )
    prob = p_0 * transitions
    return prob


def information_content_fom(
    sequence,
    transition_probs=transition_probabilities,
    alphabet_size=alphabet_size,
):
    """
    For first order Markov models

    IC(x_n|x_{n-1},...,x_1) = IC(x_n|x_{n-1}) = - log2(p(x_n | x_{n-1}))
    """
    ic = -np.log2(transition_probs[sequence[-1], sequence[-2]])
    return ic


def entropy_fom(
    sequence,
    alphabet=alphabet,
    transition_probs=transition_probabilities,
):
    """
    Entropy for first order Markov model
    """
    entropy_components = []
    for al in alphabet:
        in_seq = np.zeros_like(sequence)
        in_seq[:-1] = sequence[:-1]
        in_seq[-1] = al

        ic = information_content_fom(
            in_seq, transition_probs, alphabet_size=len(alphabet)
        )
        prob = transition_probs[al, sequence[-2]]

        entropy_components.append(ic * prob)
    entropy = np.sum(entropy_components)

    return entropy

## Tasks 4: Markov Chain Generation

1. By computing the probability $ \mathbb{P}(\mathbf{x_i}\mid \mathbf{x_{i-1}}) $ in task 3 you have fully specified a discrete-time finite state space Markov Chain model (https://en.wikipedia.org/wiki/Discrete-time_Markov_chain)! Given an initial symbol "s_0", you can generate the subsequent symbols by sampling from the conditional probability distribution

$$ p(\mathbf{x_i}\mid \mathbf{x_{i-1}} = \mathbf{s_{0}}) $$

Write a function that samples from a finite state space given an input probability distribution.

2. Use the previously defined function and the Markov Chain to write a sequence generator based on an initial symbol.

3. Start several "walkers", i.e. sampled/generated sequences. 


In [ ]:
from partitura.utils.synth import synthesize, SAMPLE_RATE
import IPython.display as ipd


def sample(distribution):
    cs = distribution.cumsum()
    samp = np.random.rand(1)
    return list(samp < cs).index(True)


def generate(start=0, length=100):
    melody = [start]
    for k in range(length - 1):
        melody.append(sample(transition_probabilities[melody[-1], :]))
    return np.array(melody)

In [ ]:
def synthesize_generated_sequence(generated_sequence):

    note_array = enc.decode(generated_sequence)

    signal = synthesize(
        note_array,
        harmonic_dist=3,  # "shepard",
    )
    ipd.display(ipd.Audio(data=signal, rate=SAMPLE_RATE, normalize=False))

    return note_array


n_notes = 100
generated_sequence = generate(length=n_notes)
print(generated_sequence)
note_array = synthesize_generated_sequence(generated_sequence)

## Tasks 5: n-gram Context Model:

1. The Markov Chains used until now have only very limited memory. In fact, they only ever know the last played pitch or duration. Longer memory models can be created by using the conditional probability of any new symbol based on an n-gram context of the symbol (https://en.wikipedia.org/wiki/N-gram):
$$ p(\mathbf{x_i}\mid \mathbf{x_{i-1}}, \dots, \mathbf{x_{i-n}}) $$

This probability will generally not look like a matrix anymore, but we can easily encode it as a dictionary. Write a function that creates a 3-gram context model from the data sequence **X**!

2. The longer the context, the more data we need to get meaningful or even existing samples for all contexts (note that the number of different contexts grows exponentially with context length). What could we do to approximate the distribution for unseen contexts?

In [ ]:
from collections import defaultdict
import copy


def create_context_model(sequences, n, n_states=alphabet_size):
    a_priori_probability = np.ones(n_states) / n_states
    context_model = defaultdict(lambda: copy.copy(a_priori_probability))
    for sequence in sequences:
        for idx in range(len(sequence) - n):
            local_string = ""
            for p in sequence[idx : idx + n]:
                local_string += str(p)
            context_model[local_string][sequence[idx + n]] += 1

    for key in context_model.keys():
        prob_dist = context_model[key]
        context_model[key] = prob_dist / prob_dist.sum()

    return context_model


cm = create_context_model(encoded_sequences, 5)

In [ ]:
def generate_with_context_model(start=[0, 0, 0], length=100, context_model=cm):
    melody = start
    for k in range(length):
        key = ""
        for p in melody[-len(start) :]:
            key += str(p)
        
        melody.append(sample(context_model[key]))
    return np.array(melody)


start = list(enc.transform([60, 62, 64]))
generated_sequence = generate_with_context_model(start=start)

note_array = synthesize_generated_sequence(generated_sequence)